# Module 2b: Strands-Evals Evaluation

## Overview

This notebook evaluates the multi-agent e-commerce customer service system using **strands-evals**.

### Optimization: Run Agent Once, Evaluate Multiple Times

Unlike the naive approach that runs the agent once per evaluator (6x per test case), this notebook:
1. **Runs the agent ONCE** per test case and caches responses
2. **Evaluates cached responses** with all metrics

This reduces agent invocations from `N_cases × N_evaluators` to just `N_cases`.

### Evaluation Metrics (6 total)
1. Goal Success
2. Helpfulness
3. Routing Accuracy
4. Policy Compliance
5. Response Quality
6. Customer Satisfaction

### Time: ~30 minutes

## Step 1: Setup

Install dependencies and create the customer service instance.

In [1]:
# Install dependencies
!pip install -q strands-agents strands-agents-evals boto3 pandas deepeval


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [2]:
import json
import os
import sys
import boto3
import pandas as pd
from datetime import datetime

# Try to load REGION from Module 1
try:
    %store -r REGION
    print(f"✓ Loaded REGION from Module 1: {REGION}")
except:
    print("Could not load REGION from Module 1, using session default")
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'

print(f"Region: {REGION}")

# Set environment variable
os.environ['AWS_REGION'] = REGION

# Always create new customer_service instance (cannot be pickled)
print("\nCreating customer service instance...")
sys.path.insert(0, '../01-multi-agent-prototype/agents')
from orchestrator import MultiAgentCustomerService
customer_service = MultiAgentCustomerService(region=REGION)
print("✓ Customer service initialized")

✓ Loaded REGION from Module 1: us-west-2
Region: us-west-2

Creating customer service instance...
✓ Customer service initialized


## Step 2: Load Evaluation Experiment

Load the test cases and convert them to the format expected by strands-evals.

In [3]:
# Import strands-evals
from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

# Load evaluation dataset
with open('evaluation_dataset.json', 'r') as f:
    eval_data = json.load(f)

print(f"Loaded {len(eval_data['test_cases'])} test cases")
print(f"\nSample test case:")
print(json.dumps(eval_data['test_cases'][0], indent=2))

Loaded 25 test cases

Sample test case:
{
  "id": "order_001",
  "category": "order_status",
  "difficulty": "easy",
  "input": "What's the status of order ORD-2024-10002?",
  "expected_agent": "order_agent",
  "expected_tools": [
    "get_order_status"
  ],
  "expected_output_contains": [
    "shipped",
    "UPS",
    "tracking"
  ],
  "ground_truth": "Order ORD-2024-10002 is shipped via UPS with tracking number 1Z999AA10123456784"
}


In [4]:
# Convert test cases to Case objects
test_cases = []
for tc in eval_data['test_cases']:
    case = Case(
        name=tc['id'],
        input=tc['input'],
        expected_output=tc.get('ground_truth', ''),
        metadata={
            'category': tc['category'],
            'expected_agent': tc.get('expected_agent'),
            'expected_tools': tc.get('expected_tools', []),
            'expected_output_contains': tc.get('expected_output_contains', [])
        }
    )
    test_cases.append(case)

print(f"Created {len(test_cases)} Case objects")
print(f"\nCategories: {set(tc.metadata['category'] for tc in test_cases)}")

Created 25 Case objects

Categories: {'out_of_scope', 'suspended_account', 'payment_methods', 'edge_case', 'address_update', 'order_return', 'product_inventory', 'order_not_found', 'order_history', 'product_policy', 'order_status', 'password_reset', 'order_tracking', 'order_cancellation', 'membership_benefits', 'refund_status', 'multi_agent', 'product_comparison', 'product_recommendation', 'product_details', 'product_search', 'account_info'}


## Step 3: Run Agent Once and Cache Responses

**Key Optimization**: Run the agent ONCE for each test case and cache responses. This avoids running the agent 6x per test case (once per evaluator).

In [ ]:
import time

# Cache to store agent responses (key: case name, value: response)
response_cache = {}

def run_agent_and_cache(cases: list, customer_service) -> dict:
    """
    Run the agent ONCE for each test case and cache responses.
    
    Args:
        cases: List of Case objects to evaluate
        customer_service: The multi-agent system
        
    Returns:
        dict: Mapping of case name to response
    """
    print(f"Running agent on {len(cases)} test cases (ONE TIME ONLY)...\n")
    
    for i, case in enumerate(cases):
        print(f"[{i+1}/{len(cases)}] {case.name} ({case.metadata['category']})")
        print(f"    Query: {case.input[:60]}...")
        
        start_time = time.time()
        try:
            response = customer_service.chat(case.input)
            latency = time.time() - start_time
            response_cache[case.name] = str(response)
            print(f"    Latency: {latency:.1f}s")
            print(f"    Response: {str(response)[:100]}...")
        except Exception as e:
            response_cache[case.name] = f"Error: {str(e)}"
            print(f"    ERROR: {str(e)[:50]}")
        print()
        time.sleep(0.5)  # Rate limiting
    
    return response_cache


def cached_task(case) -> str:
    """
    Task function that returns cached response instead of calling agent.
    This allows running multiple evaluators without re-invoking the agent.
    """
    return response_cache.get(case.name, "Error: Response not found in cache")


# Run agent on first 5 test cases and cache responses
selected_cases = test_cases[:5]
response_cache = run_agent_and_cache(selected_cases, customer_service)

print(f"\n✓ Cached {len(response_cache)} responses")
print("✓ All evaluators will use cached responses (no additional agent calls)")

## Step 4: Run Evaluators on Cached Responses

Now run all 6 evaluators using the **cached responses**. No additional agent calls are made.

**Note:** Each Experiment uses `cached_task` which returns pre-computed responses.

In [6]:
# Create Goal Success Evaluator
goal_success_evaluator = OutputEvaluator(
    rubric="""Evaluate if the agent response successfully addresses the customer's request.

Score 1.0: The response fully addresses the customer's request with accurate, helpful information
Score 0.75: The response mostly addresses the request but may be missing minor details
Score 0.5: The response partially addresses the request but has significant gaps
Score 0.25: The response attempts to address the request but fails to provide useful help
Score 0.0: The response does not address the request at all or provides incorrect information

Respond with a JSON object containing 'score' (float 0-1) and 'reasoning' (string)."""
)

print("Goal Success Evaluator created")

Goal Success Evaluator created


In [ ]:
# Run Goal Success Evaluation using CACHED responses
goal_success_experiment = Experiment(
    cases=selected_cases,
    evaluators=[goal_success_evaluator]
)

print("Running goal success evaluation (using cached responses)...")
goal_success_results = goal_success_experiment.run_evaluations(cached_task)

In [ ]:
# Extract the report (first and only item in results list)
goal_success_report = goal_success_results[0]

print("\n" + "="*60)
print("GOAL SUCCESS EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(selected_cases, goal_success_report.scores, goal_success_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Goal Success Score: {goal_success_report.overall_score:.2f}")
print(f"Pass Rate: {sum(goal_success_report.test_passes)}/{len(goal_success_report.test_passes)}")
print(f"{'='*60}")

In [ ]:
# Create Helpfulness Evaluator
helpfulness_evaluator = OutputEvaluator(
    rubric="""Evaluate how helpful the agent response is for the customer.

Score 1.0: Extremely helpful - provides clear, actionable information and anticipates follow-up needs
Score 0.75: Very helpful - provides good information that addresses the customer's needs
Score 0.5: Somewhat helpful - provides basic information but could be more detailed
Score 0.25: Minimally helpful - provides limited useful information
Score 0.0: Not helpful - does not provide any useful information

Respond with a JSON object containing 'score' (float 0-1) and 'reasoning' (string)."""
)

# Run Helpfulness Evaluation using CACHED responses
helpfulness_experiment = Experiment(
    cases=selected_cases,
    evaluators=[helpfulness_evaluator]
)

print("Running helpfulness evaluation (using cached responses)...")
helpfulness_results = helpfulness_experiment.run_evaluations(cached_task)

# Extract the report
helpfulness_report = helpfulness_results[0]

print("\n" + "="*60)
print("HELPFULNESS EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(selected_cases, helpfulness_report.scores, helpfulness_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Helpfulness Score: {helpfulness_report.overall_score:.2f}")
print(f"Pass Rate: {sum(helpfulness_report.test_passes)}/{len(helpfulness_report.test_passes)}")
print(f"{'='*60}")

## Step 5: Custom Evaluators

Run domain-specific custom evaluators for routing accuracy, policy compliance, and response quality.

In [10]:
  # Reload custom evaluators module (in case it was already imported)
  import importlib
  import sys
  if 'custom_evaluators' in sys.modules:
      import custom_evaluators
      importlib.reload(custom_evaluators)

In [11]:
# Import custom evaluators
from custom_evaluators import (
    RoutingAccuracyEvaluator,
    PolicyComplianceEvaluator,
    ResponseQualityEvaluator,
    CustomerSatisfactionEvaluator
)

print("Custom evaluators imported")

Custom evaluators imported


In [12]:
# Routing Accuracy Evaluation
routing_evaluator = RoutingAccuracyEvaluator()
routing_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[routing_evaluator]  # Pass as list
)

print("Running routing accuracy evaluation...")
routing_results = routing_experiment.run_evaluations(run_customer_service_task)

# Extract the report
routing_report = routing_results[0]

print("\n" + "="*60)
print("ROUTING ACCURACY EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], routing_report.scores, routing_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Routing Accuracy Score: {routing_report.overall_score:.2f}")
print(f"Pass Rate: {sum(routing_report.test_passes)}/{len(routing_report.test_passes)}")
print(f"{'='*60}")

Running routing accuracy evaluation...

Tool #12: delegate_to_order_agent

Tool #12: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number

In [15]:
# Policy Compliance Evaluation
policy_evaluator = PolicyComplianceEvaluator()
policy_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[policy_evaluator]  # Pass as list
)

print("Running policy compliance evaluation...")
policy_results = policy_experiment.run_evaluations(run_customer_service_task)

# Extract the report
policy_report = policy_results[0]

print("\n" + "="*60)
print("POLICY COMPLIANCE EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], policy_report.scores, policy_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Policy Compliance Score: {policy_report.overall_score:.2f}")
print(f"Pass Rate: {sum(policy_report.test_passes)}/{len(policy_report.test_passes)}")
print(f"{'='*60}")

Running policy compliance evaluation...
I'll check the status of order ORD-2024-10002 for you.
Tool #22: delegate_to_order_agent

Tool #22: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shippin

In [18]:
# Response Quality Evaluation
quality_evaluator = ResponseQualityEvaluator()
quality_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[quality_evaluator]  # Pass as list
)

print("Running response quality evaluation...")
quality_results = quality_experiment.run_evaluations(run_customer_service_task)

# Extract the report
quality_report = quality_results[0]

print("\n" + "="*60)
print("RESPONSE QUALITY EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], quality_report.scores, quality_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Response Quality Score: {quality_report.overall_score:.2f}")
print(f"Pass Rate: {sum(quality_report.test_passes)}/{len(quality_report.test_passes)}")
print(f"{'='*60}")

Running response quality evaluation...
I'll check the status of order ORD-2024-10002 for you.
Tool #27: delegate_to_order_agent

Tool #27: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping

In [19]:
# Customer Satisfaction Evaluation
satisfaction_evaluator = CustomerSatisfactionEvaluator()
satisfaction_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[satisfaction_evaluator]  # Pass as list
)

print("Running customer satisfaction evaluation...")
satisfaction_results = satisfaction_experiment.run_evaluations(run_customer_service_task)

# Extract the report
satisfaction_report = satisfaction_results[0]

print("\n" + "="*60)
print("CUSTOMER SATISFACTION EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], satisfaction_report.scores, satisfaction_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Customer Satisfaction Score: {satisfaction_report.overall_score:.2f}")
print(f"Pass Rate: {sum(satisfaction_report.test_passes)}/{len(satisfaction_report.test_passes)}")
print(f"{'='*60}")

Running customer satisfaction evaluation...
I'll check the status of order ORD-2024-10002 for you.
Tool #32: delegate_to_order_agent

Tool #32: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shi

## Step 6: Extract and Analyze Results

Collect scores from all evaluations and compute baseline metrics.

In [20]:
# Helper function to extract scores from EvaluationReport
def extract_scores_from_report(report):
    """Extract scores from evaluation report"""
    return report.scores

# Extract all scores
goal_success_scores = extract_scores_from_report(goal_success_report)
helpfulness_scores = extract_scores_from_report(helpfulness_report)
routing_scores = extract_scores_from_report(routing_report)
policy_scores = extract_scores_from_report(policy_report)
quality_scores = extract_scores_from_report(quality_report)
satisfaction_scores = extract_scores_from_report(satisfaction_report)

print("✓ Scores extracted from all evaluations")
print(f"\nGoal Success: {goal_success_scores}")
print(f"Helpfulness: {helpfulness_scores}")
print(f"Routing: {routing_scores}")
print(f"Policy: {policy_scores}")
print(f"Quality: {quality_scores}")
print(f"Satisfaction: {satisfaction_scores}")

✓ Scores extracted from all evaluations

Goal Success: [1.0, 1.0, 0.0, 0.0, 1.0]
Helpfulness: [1.0, 1.0, 0.75, 1.0, 1.0]
Routing: [1.0, 1.0, 1.0, 1.0, 1.0]
Policy: [1.0, 1.0, 1.0, 0.0, 1.0]
Quality: [1.0, 1.0, 0.8, 0.2, 0.9]
Satisfaction: [1.0, 1.0, 0.25, 0.75, 1.0]


In [21]:
# Create DataFrame with all results
results_df = pd.DataFrame({
    'test_case': [case.name for case in test_cases[:5]],
    'category': [case.metadata['category'] for case in test_cases[:5]],
    'goal_success': goal_success_scores,
    'helpfulness': helpfulness_scores,
    'routing_accuracy': routing_scores,
    'policy_compliance': policy_scores,
    'response_quality': quality_scores,
    'customer_satisfaction': satisfaction_scores
})

print("\nEvaluation Results DataFrame:")
print(results_df.to_string(index=False))


Evaluation Results DataFrame:
test_case           category  goal_success  helpfulness  routing_accuracy  policy_compliance  response_quality  customer_satisfaction
order_001       order_status           1.0         1.00               1.0                1.0               1.0                   1.00
order_002     order_tracking           1.0         1.00               1.0                1.0               1.0                   1.00
order_003       order_return           0.0         0.75               1.0                1.0               0.8                   0.25
order_004 order_cancellation           0.0         1.00               1.0                0.0               0.2                   0.75
order_005      order_history           1.0         1.00               1.0                1.0               0.9                   1.00


## Step 7: Baseline Metrics

Calculate average scores to establish baseline performance.

In [22]:
# Calculate baseline metrics
baseline_metrics = {
    'timestamp': datetime.now().isoformat(),
    'total_test_cases': len(results_df),
    'goal_success_rate': results_df['goal_success'].mean(),
    'helpfulness': results_df['helpfulness'].mean(),
    'routing_accuracy': results_df['routing_accuracy'].mean(),
    'policy_compliance': results_df['policy_compliance'].mean(),
    'response_quality': results_df['response_quality'].mean(),
    'customer_satisfaction': results_df['customer_satisfaction'].mean()
}

print("\n" + "="*60)
print("BASELINE METRICS")
print("="*60)
for metric, value in baseline_metrics.items():
    if isinstance(value, float) and value <= 1.0:
        print(f"{metric:.<40} {value:.1%}")
    else:
        print(f"{metric:.<40} {value}")


BASELINE METRICS
timestamp............................... 2026-01-20T14:56:26.041788
total_test_cases........................ 5
goal_success_rate....................... 60.0%
helpfulness............................. 95.0%
routing_accuracy........................ 100.0%
policy_compliance....................... 80.0%
response_quality........................ 78.0%
customer_satisfaction................... 80.0%


In [23]:
# Performance by category
print("\n" + "="*60)
print("PERFORMANCE BY CATEGORY")
print("="*60)

category_metrics = results_df.groupby('category')[[
    'goal_success', 'helpfulness', 'routing_accuracy', 
    'policy_compliance', 'response_quality', 'customer_satisfaction'
]].mean()

print(category_metrics.to_string())


PERFORMANCE BY CATEGORY
                    goal_success  helpfulness  routing_accuracy  policy_compliance  response_quality  customer_satisfaction
category                                                                                                                   
order_cancellation           0.0         1.00               1.0                0.0               0.2                   0.75
order_history                1.0         1.00               1.0                1.0               0.9                   1.00
order_return                 0.0         0.75               1.0                1.0               0.8                   0.25
order_status                 1.0         1.00               1.0                1.0               1.0                   1.00
order_tracking               1.0         1.00               1.0                1.0               1.0                   1.00


## Step 8: Define Production Thresholds

Set alert thresholds based on baseline metrics (typically 80-90% of baseline).

In [24]:
# Define production thresholds
production_thresholds = {
    'goal_success_rate': 0.70,       # Alert if below 70%
    'helpfulness': 0.65,              # Alert if below 65%
    'routing_accuracy': 0.75,         # Alert if below 75%
    'policy_compliance': 0.80,        # Alert if below 80%
    'response_quality': 0.65,         # Alert if below 65%
    'customer_satisfaction': 0.70     # Alert if below 70%
}

print("\n" + "="*60)
print("PRODUCTION THRESHOLDS")
print("="*60)
for metric, threshold in production_thresholds.items():
    print(f"{metric:.<40} {threshold:.0%}")


PRODUCTION THRESHOLDS
goal_success_rate....................... 70%
helpfulness............................. 65%
routing_accuracy........................ 75%
policy_compliance....................... 80%
response_quality........................ 65%
customer_satisfaction................... 70%


In [25]:
# Check current performance against thresholds
print("\n" + "="*60)
print("THRESHOLD VALIDATION")
print("="*60)

all_pass = True
for metric, threshold in production_thresholds.items():
    current = baseline_metrics.get(metric, 0)
    passed = current >= threshold
    
    status = "✓ PASS" if passed else "✗ FAIL"
    
    if not passed:
        all_pass = False
    
    print(f"[{status}] {metric}: {current:.1%} (threshold: {threshold:.0%})")

print("\n" + "="*60)
if all_pass:
    print("✓ ALL THRESHOLDS PASSED - Ready for production!")
else:
    print("⚠ SOME THRESHOLDS FAILED - Review before production")
print("="*60)


THRESHOLD VALIDATION
[✗ FAIL] goal_success_rate: 60.0% (threshold: 70%)
[✓ PASS] helpfulness: 95.0% (threshold: 65%)
[✓ PASS] routing_accuracy: 100.0% (threshold: 75%)
[✓ PASS] policy_compliance: 80.0% (threshold: 80%)
[✓ PASS] response_quality: 78.0% (threshold: 65%)
[✓ PASS] customer_satisfaction: 80.0% (threshold: 70%)

⚠ SOME THRESHOLDS FAILED - Review before production


## Step 9: Save Results

Store evaluation results and baseline metrics for Module 3.

In [26]:
# Save detailed results
results_df.to_csv('evaluation_results.csv', index=False)
print("✓ Saved detailed results to evaluation_results.csv")

# Save baseline metrics
with open('baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2, default=str)
print("✓ Saved baseline metrics to baseline_metrics.json")

# Store for next module
%store baseline_metrics
%store production_thresholds
%store REGION
print("\n✓ Data stored for Module 3!")

✓ Saved detailed results to evaluation_results.csv
✓ Saved baseline metrics to baseline_metrics.json
Stored 'baseline_metrics' (dict)
Stored 'production_thresholds' (dict)
Stored 'REGION' (str)

✓ Data stored for Module 3!


---

## Module 2 Complete!

You have successfully:

1. **Part 1 (strands-evals)**: Ran quick interactive evaluation with custom evaluators
2. **Part 2 (DeepEval)**: Ran comprehensive batch evaluation with experiment tracking

### Files Created

| File | Description |
|------|-------------|
| `evaluation_results.csv` | strands-eval per-case results |
| `baseline_metrics.json` | strands-eval aggregate metrics |
| `experiments/{id}/` | Full experiment with agent outputs, eval results |

### Next Steps

Based on your **Production Readiness Decision**:

- **READY FOR PRODUCTION** → Proceed to **Module 3: Production Deployment**
- **NEEDS IMPROVEMENT** → Review failing cases, improve agent, then re-run evaluation

### The Evaluation Lifecycle

```
Stage 1 (Prototype)     Stage 2 (Production)     Stage 3 (Improvement)
      │                        │                        │
      ▼                        ▼                        ▼
   DeepEval              strands-eval/             DeepEval
   Offline Eval         AgentCore Online          Re-evaluate
      │                        │                        │
      ▼                        ▼                        ▼
   Go/No-Go              Monitor & Alert         Validate Fix
```